# 00 — v15를 이해하기 위한 토이 프로젝트: 개념과 아키텍처

신입 입장에서 "v15 모델의 원리를 깨우친다"는 건 코드 한 줄 읽는 게 아니라, **지금 공개적으로
알려진 end-to-end 자율주행 스택의 세 가지 축**을 몸으로 체득하는 것에 가깝습니다.

1. **멀티 카메라 서라운드 퓨전** — 여러 대의 카메라(전방/측방/후방 등)가 찍은, 서로 다른
   시점의 이미지를 하나의 "세계에 대한 표현(representation)"으로 합치는 것.
2. **시계열(시퀀스) 모델링** — 한 프레임이 아니라 과거 N프레임 + 과거 행동(조향/가속) 의
   흐름을 누적해서 "지금 상황"을 추정하는 것. 정적인 이미지 분류와 가장 크게 다른 지점입니다.
3. **월드 모델(World Model)** — "지금까지 본 것 + 앞으로 할 행동"으로부터 "앞으로 무슨 일이
   일어날지"를 예측(상상, imagination)하는 모델. 플래닝/시뮬레이션에 쓰이는 핵심 구성요소입니다.

업계에서 공개적으로 논의되는 최신 end-to-end 스택들의 공통된 방향은 대략 이렇습니다
(특정 회사의 비공개 내부 구현이 아니라, 학계/업계에 널리 알려진 설계 원칙 수준의 이야기입니다):

- 모듈을 여러 개로 쪼개 손으로 규칙을 짜는 대신, **카메라 입력 → 제어 출력까지 하나의 신경망**으로
  최대한 엮는다 ("photon-to-control"에 가까운 방향).
- 명시적인 HD맵/차선 그래프 대신, **신경망이 스스로 점유 공간(occupancy)과 시간에 따른
  변화를 예측**하도록 학습시킨다.
- 학습된 표현 위에서 **여러 미래를 "상상"해보고 그 중 안전한 경로를 고르는 것**이 플래닝의
  핵심이 된다. 이게 바로 "월드 모델"이 필요한 이유입니다.

## 이 프로젝트에서 실제로 만드는 것

RTX 2060(6GB)에서 돌아가는, **완전히 synthetic(인공적으로 만든)한 toy 환경**을 만들고
그 위에서 위 세 가지 축을 모두 직접 구현해봅니다.

- 진짜 Tesla 코드/데이터가 아닙니다. 공개된 원리를 "작게, 손으로 다 구현해서" 체득하는
  학습용 프로젝트입니다.
- 작은 2D 주행 시뮬레이터가 "정답 세계(world)"를 만들고, 그 세계를 3대의 가상 카메라
  (front / left / right)가 서로 다른 각도로 찍습니다.
- 시뮬레이터가 만든 (멀티카메라 영상, 조향/가속 행동) 시퀀스로 **재귀적(recurrent) 월드 모델**을
  학습시키고, 마지막 노트북에서 그 모델을 "눈을 감고" 혼자 상상하게 시켜본 뒤 (observation 없이
  roll-forward), 어디서부터 틀리기 시작하는지를 직접 찾아봅니다 — 이게 실무에서 "이슈를 발굴"
  하는 작업의 축소판입니다.

## 노트북 구성

| 노트북 | 내용 |
|---|---|
| `01_sim_and_data.ipynb` | 시뮬레이터 + 멀티카메라 렌더링 + Dataset/DataLoader |
| `02_model_architecture.ipynb` | 인코더/재귀 월드모델/디코더 아키텍처, 파라미터 수 확인 |
| `03_train_world_model.ipynb` | 학습 루프, loss curve, 체크포인트 저장 |
| `04_imagination_rollout_and_issue_discovery.ipynb` | observation 없이 상상 롤아웃 → 오차 누적 시각화 → 이슈 리포트 작성 |

공유 코드는 전부 `wm_toy.py` 한 파일에 있습니다 (다른 폴더의 `video_utils.py`와 같은 패턴).


In [1]:
import torch
import wm_toy as W

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

model = W.WorldModel()
print(f"world model parameters: {model.num_params():,}")
print(f"encoder params:  {sum(p.numel() for p in model.encoder.parameters()):,}")
print(f"decoder params:  {sum(p.numel() for p in model.decoder.parameters()):,}")
print(f"gru+head params: {sum(p.numel() for p in model.gru.parameters()) + sum(p.numel() for p in model.embed_head.parameters()):,}")


torch: 2.15.0+cu132 | cuda available: False
world model parameters: 1,025,411
encoder params:  90,992
decoder params:  817,971
gru+head params: 116,448


### 스케일 감각

위에서 보듯 전체 모델은 대략 1백만 파라미터 수준입니다. 실제 양산 end-to-end 스택은 수십~수백억
파라미터, 수천 시간 분량의 실제 주행 영상으로 학습됩니다. 여기서 중요한 건 **크기가 아니라
"관(pipe)의 모양"**입니다 — 멀티카메라 인코더 → 시간 축으로 누적되는 은닉 상태 → 그 상태에서
관측 없이도 다음을 예측하는 구조. 이 모양 자체를 손으로 짜보고 나면, 실제 대규모 스택의
논문/블로그/토크를 읽을 때 "아, 이 부분이 그 역할이구나"를 훨씬 빠르게 알아볼 수 있습니다.

다음 노트북(`01_sim_and_data.ipynb`)에서 시뮬레이터부터 시작합니다.
